# Task 3 - Step 0: Verify the shared protocol, the ERM checkpoint and Sketch isolation

Task 3 must use **exactly** Task 2's protocol (splits, initialisation, head, preprocessing, augmentation,
domain-balanced sampling, optimiser, epoch budget, early stopping, seed), and its ERM is the Task 2 Source-only
checkpoint, loaded rather than retrained. This notebook checks both, and also:

* fixes the **sharpness batch** (32 source-validation images per source domain, seed 6304) before any model is
  trained;
* statically scans the Task 3 training/diagnostic code (notebooks 00-02, `task3/methods/`, `shared/training.py`)
  to confirm none of it calls a Sketch loader. Sketch may only be loaded by notebook 03.

**No Sketch image or label is loaded here.**

In [1]:
# ---- Task 3 common header (identical in every Task 3 notebook) ----
# NOTE: this header never loads any Sketch image or label. Only notebook 03 does, after the freeze check.
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "shared" / "pacs.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the PA1 repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared import pacs, pacs_protocol as proto
from shared.config import load_config

T3 = REPO / "task3"
CFG_DIR = T3 / "configs"
EVAL_CFG = __import__("yaml").safe_load((CFG_DIR / "evaluation.yaml").read_text())
SEED = 6304
# Smoke mode (env TASK3_SMOKE=1): 2 epochs x 5 updates per trained run, outputs under _smoke/, and notebook 03 uses
# RANDOM stand-in images and labels, so no Sketch image or label is touched.
SMOKE = os.environ.get("TASK3_SMOKE", "0") == "1"
SUB = "_smoke" if SMOKE else ""
RES = T3 / "results" / SUB
TAB, FIG = RES / "tables", RES / "figures"
DATA_TAB = T3 / "results" / "tables"      # protocol/data-prep files from notebook 00 (same in smoke and real mode)
DATA_TAB.mkdir(parents=True, exist_ok=True)
CKPT = T3 / "checkpoints" / SUB            # git-ignored
CACHE = T3 / "cache" / SUB                 # git-ignored
for p in [TAB, FIG, CKPT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ERM_DIR = REPO / "task2" / "checkpoints" / "source_only"      # ERM = Task 2 Source-only, loaded, never retrained
TRAIN_RUNS = ["dan_dg", "sam", "dan_dg_lambda0.1", "dan_dg_lambda10"]
MAIN_RUNS = ["erm", "dan_dg", "sam"]
STUDY_RUNS = ["dan_dg_lambda0.1", "dan_dg", "dan_dg_lambda10"]    # controlled study: lambda_DG in {0.1, 1, 10}
ALL_RUNS = ["erm", "dan_dg", "sam", "dan_dg_lambda0.1", "dan_dg_lambda10"]
RUN_NAME = {c: load_config(CFG_DIR, c)["run_name"] for c in ALL_RUNS}
LABEL = {"erm": "ERM", "dan_dg": "DAN-DG (λ=1)", "sam": "SAM (ρ=0.05)",
         "dan_dg_lambda0.1": "DAN-DG (λ=0.1)", "dan_dg_lambda10": "DAN-DG (λ=10)"}


def run_dir(n):
    return ERM_DIR if n == "erm" else CKPT / RUN_NAME[n]


def load_trained(n):
    """Model with the selected checkpoint of run ``n`` (ERM: the Task 2 Source-only checkpoint)."""
    from shared.models import build_model
    ck = torch.load(run_dir(n) / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(7, SEED)
    model.load_state_dict(ck["model"])
    return model.to(DEVICE).eval(), ck


plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 9, "savefig.dpi": 150})


def savefig(fig, stem):
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    print("saved figure", stem)


print("REPO:", REPO, "| device:", DEVICE, "| smoke:", SMOKE)

REPO: C:\Users\afifh\Desktop\ATML\PA1 | device: cuda | smoke: False


In [2]:
import yaml
b2 = yaml.safe_load((REPO / "task2" / "configs" / "base.yaml").read_text())
b3 = yaml.safe_load((CFG_DIR / "base.yaml").read_text())
assert b2 == b3, "Task 3 base protocol must be identical to Task 2's"
split = proto.load_splits()
print("protocol identical to Task 2:", b2 == b3)
print("split file:", proto.SPLIT_PATH.relative_to(REPO), "| sha256", pacs.file_sha256(proto.SPLIT_PATH)[:16], "...")
print({d: (v["n_train"], v["n_val"]) for d, v in split["domains"].items()})

protocol identical to Task 2: True
split file: shared\splits\pacs_sketch_seed6304.json | sha256 153c58c995e7ccaf ...
{'photo': (1336, 334), 'art_painting': (1638, 410), 'cartoon': (1875, 469)}


## ERM = Task 2 Source-only checkpoint (loaded, not retrained)

In [3]:
erm_summary = json.loads((ERM_DIR / "summary.json").read_text())
erm_cfg = yaml.safe_load((ERM_DIR / "config.yaml").read_text())
assert erm_cfg["method"] == "source_only" and erm_cfg["seed"] == SEED and not erm_summary["uses_target_images"]
erm_ref = {"checkpoint": str((ERM_DIR / "best.pt").relative_to(REPO)).replace("\\", "/"),
           "checkpoint_sha256": pacs.file_sha256(ERM_DIR / "best.pt"), "best_epoch": erm_summary["best_epoch"],
           "best_mean_val_macro_f1": erm_summary["best_mean_val_macro_f1"], "uses_target_images": erm_summary["uses_target_images"],
           "train_config_equals_task3_base": {k: erm_cfg[k] == b3[k] for k in ("seed", "data", "model", "train")}}
assert all(erm_ref["train_config_equals_task3_base"].values())
(DATA_TAB / "task3_erm_reference.json").write_text(json.dumps(erm_ref, indent=1))
erm_ref

{'checkpoint': 'task2/checkpoints/source_only/best.pt',
 'checkpoint_sha256': '6245ad4a87c63c62c1aa6e299212a81c1717fd73159485ced5ac2157798eb825',
 'best_epoch': 6,
 'best_mean_val_macro_f1': 0.9316911755048768,
 'uses_target_images': False,
 'train_config_equals_task3_base': {'seed': True,
  'data': True,
  'model': True,
  'train': True}}

## Fixed sharpness batch (32 per source validation domain, seed 6304)

In [4]:
rng = np.random.default_rng(SEED)
k = EVAL_CFG["sharpness"]["per_source_domain"]
sharp_batch = {d: sorted(int(i) for i in rng.choice(split["domains"][d]["n_val"], k, replace=False)) for d in pacs.SOURCE_DOMAINS}
(DATA_TAB / "task3_sharpness_batch.json").write_text(json.dumps({"seed": SEED, "per_source_domain": k,
    "indices_into_source_val": sharp_batch, "radius": EVAL_CFG["sharpness"]["radius"]}, indent=1))
{d: v[:6] for d, v in sharp_batch.items()}

{'photo': [0, 2, 49, 55, 64, 67],
 'art_painting': [14, 17, 29, 35, 54, 87],
 'cartoon': [71, 73, 80, 88, 104, 116]}

## Static check: no Sketch access in Task 3 training / diagnostic code

In [5]:
import re
pattern = re.compile(r"load_" + r"target_\w+\s*\(")          # split so this cell does not match itself
checked, hits = [], []
for nb in ["00_verify_protocol.ipynb", "01_train_methods.ipynb", "02_source_diagnostics.ipynb"]:
    for c in json.loads((T3 / nb).read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code":
            for line in "".join(c["source"]).splitlines():
                if pattern.search(line):
                    hits.append(f"{nb}: {line.strip()}")
    checked.append(nb)
for py in list((T3 / "methods").glob("*.py")) + [REPO / "shared" / "training.py"]:
    for line in py.read_text(encoding="utf-8").splitlines():
        if pattern.search(line):
            hits.append(f"{py.relative_to(REPO)}: {line.strip()}")
    checked.append(str(py.relative_to(REPO)))
result = {"checked": checked, "sketch_loader_calls_found": hits, "ok": not hits}
(DATA_TAB / "task3_no_sketch_access_check.json").write_text(json.dumps(result, indent=1))
assert not hits, hits
print("no Sketch loader call in", len(checked), "Task 3 training/diagnostic files")

no Sketch loader call in 8 Task 3 training/diagnostic files
